# MapReduce with PySpark (RDD-first)

Companion notebook for **Tutorial 1** (`mapreduce-pyspark-tutorial.md`). 

Before running this notebook, generate the sample retail data once:

```bash
python3 generate_retail_data.py
```

This writes `data/customers.csv`, `data/products.csv`, `data/orders.csv`, `data/items.csv` next to this notebook (deterministic, seed=7). Keeping data generation as a separate script — instead of a notebook cell — keeps the notebook focused on MapReduce patterns rather than data plumbing.

The notebook follows the same narrative as the slide deck:

1. Word count — the canonical MapReduce example, plus three variants.

2. A synthetic **retail dataset** (customers, products, orders, items).

3. Three business questions solved with RDDs (join, composite keys, top-N).

4. Performance patterns: broadcast variables, `reduceByKey` vs `groupByKey`, accumulators, caching.

5. The same two examples redone with **DataFrames**, for comparison.


## Setup — SparkSession and SparkContext

In [1]:
import os
from pyspark.sql import SparkSession

# avoid the hostname/loopback warning
os.environ.setdefault("SPARK_LOCAL_IP", "127.0.0.1")  

# Create a SparkSession object:
spark = (SparkSession.builder
            .appName("mapreduce-tutorial")
            .master("local[*]")                       # use all local cores
            .config("spark.sql.shuffle.partitions", "4")
            .getOrCreate())

# Create SparkContext object for RDD work
sc = spark.sparkContext                                
sc.setLogLevel("WARN")

spark

## Part 1 — Word Count (classic MapReduce)

Same three steps every time: <br>

* split lines into words (`flatMap`), 
* tag each with a 1 (`map`), 
* sum the 1s by word (`reduceByKey`).

In [2]:
lines = sc.parallelize([
    "the quick brown fox",
    "the lazy dog",
    "the quick fox jumps over the lazy dog",
])

counts = (lines
    .flatMap(lambda line: line.split())     # one record -> many words
    .map(lambda w: (w, 1))                  # (word, 1)
    .reduceByKey(lambda a, b: a + b))       # sum 1s by word

for word, n in sorted(counts.collect(), key=lambda kv: -kv[1]):
    print(f"{word:10s} {n}")

the        4
quick      2
fox        2
lazy       2
dog        2
over       1
brown      1
jumps      1


### Inspect each stage with `take`

Staging the pipeline and using `take(n)` is the basic debugging loop — much safer than `collect()` on real-sized data.

In [3]:
step1 = lines.flatMap(lambda l: l.split())
step2 = step1.map(lambda w: (w, 1))
step3 = step2.reduceByKey(lambda a, b: a + b)

print("step1:", step1.take(4))
print("step2:", step2.take(4))
print("step3:", step3.collect())

step1: ['the', 'quick', 'brown', 'fox']
step2: [('the', 1), ('quick', 1), ('brown', 1), ('fox', 1)]
step3: [('the', 4), ('over', 1), ('brown', 1), ('quick', 2), ('fox', 2), ('jumps', 1), ('lazy', 2), ('dog', 2)]


### Variant 1 — top-N words

`takeOrdered` is *bounded*: it returns only N items to the driver regardless of RDD size, using a heap per partition merged at the end.

> Don't use `sortBy(...).take(n)` for top-N on large data — that sorts the *whole* dataset first.

In [4]:
top5 = counts.takeOrdered(5, key=lambda kv: -kv[1])
for word, n in top5:
    print(f"{word:10s} {n}")

the        4
quick      2
fox        2
lazy       2
dog        2


### Variant 2 — case folding + stop words

In [5]:
STOP = {"the", "a", "an", "of", "and", "to", "in", "on", "for"}

filtered_counts = (lines
    .flatMap(lambda l: l.lower().split())                  # case fold
    .filter(lambda w: w not in STOP)                       # drop stop words
    .map(lambda w: (w, 1))
    .reduceByKey(lambda a, b: a + b))

sorted(filtered_counts.collect(), key=lambda kv: -kv[1])

[('quick', 2),
 ('fox', 2),
 ('lazy', 2),
 ('dog', 2),
 ('over', 1),
 ('brown', 1),
 ('jumps', 1)]

### Variant 3 — bigrams (n-grams)

Same shape as word count — only the *map* changes.

In [6]:
def to_bigrams(line):
    tokens = line.lower().split()
    return zip(tokens, tokens[1:])           # adjacent pairs

bigrams = (lines
    .flatMap(to_bigrams)                     # ('the','quick'), ('quick','brown'), ...
    .map(lambda pair: (pair, 1))
    .reduceByKey(lambda a, b: a + b))

sorted(bigrams.collect())[:5]

[(('brown', 'fox'), 1),
 (('fox', 'jumps'), 1),
 (('jumps', 'over'), 1),
 (('lazy', 'dog'), 2),
 (('over', 'the'), 1)]

## Part 2 — Loading the retail dataset

Four related tables (already generated by `generate_retail_data.py` — see the note at the top of this notebook):

| File | Columns |
|---|---|
| `customers.csv` | `customer_id, country, segment` |
| `products.csv`  | `product_id, sku, category, price` |
| `orders.csv`    | `order_id, customer_id, order_date, channel` |
| `items.csv`     | `order_id, product_id, quantity, unit_price` |

This is the same dataset used later in the course's medallion (Bronze/Silver/Gold) unit.

In [7]:
ROOT = "data"

if not os.path.exists(f"{ROOT}/items.csv"):
    raise FileNotFoundError(
        "Sample data not found. Run 'python3 generate_retail_data.py' "
        "from this notebook's directory first."
    )

### Loading the retail data into RDDs

`first()` + `filter` is the classic header-stripping idiom on RDDs (DataFrames do this for us via `header=True`).

In [8]:
def load_csv(path):
    rdd = sc.textFile(path)
    header = rdd.first()
    return (rdd
        .filter(lambda line: line != header)
        .map(lambda line: line.split(",")))

customers = load_csv(f"{ROOT}/customers.csv")  # [cid, country, segment]
products  = load_csv(f"{ROOT}/products.csv")   # [pid, sku, category, price]
orders    = load_csv(f"{ROOT}/orders.csv")     # [oid, cid, date, channel]
items     = load_csv(f"{ROOT}/items.csv")      # [oid, pid, qty, price]

print("customers:", customers.count(), "products:", products.count())
print("orders:   ", orders.count(),    "items:   ", items.count())

customers: 200 products: 50
orders:    1000 items:    2455


## Part 3 — Three business questions

### Q1 — Total revenue by category

Each item line is `[oid, pid, qty, price]`. Join with `products` to get the category, then sum `qty x price` by category.

Notice the **two re-keys**: first on `pid` (to join), then on `category` (to aggregate). Every `reduceByKey` and every `join` is a shuffle — re-keying is how you steer it.

In [9]:
pid_to_cat = products.map(lambda r: (int(r[0]), r[2]))                    # (pid, category)
item_kv    = items.map(lambda r: (int(r[1]), int(r[2]) * float(r[3])))    # (pid, line_total)

joined = item_kv.join(pid_to_cat)                                          # (pid, (line_total, category))

by_cat = (joined
    .map(lambda kv: (kv[1][1], kv[1][0]))                                 # (category, line_total)
    .reduceByKey(lambda a, b: a + b))

for cat, rev in sorted(by_cat.collect(), key=lambda kv: -kv[1]):
    print(f"  {cat:12s} ${rev:>10,.2f}")

  Electronics  $258,738.93
  Books        $209,723.78
  Beauty       $198,739.59
  Home         $143,098.98
  Apparel      $100,759.15


### Q2 — Top 5 customers by total spend

In [10]:
oid_total = items.map(lambda r: (r[0], int(r[2]) * float(r[3])))   # (oid, line_total)
oid_cid   = orders.map(lambda r: (r[0], int(r[1])))                # (oid, cid)

spend_per_customer = (oid_total
    .join(oid_cid)                              # (oid, (line_total, cid))
    .map(lambda kv: (kv[1][1], kv[1][0]))        # (cid, line_total)
    .reduceByKey(lambda a, b: a + b))

top5 = spend_per_customer.takeOrdered(5, key=lambda kv: -kv[1])   # safe at any scale
for cid, total in top5:
    print(f"  customer {cid:>3}  ${total:>10,.2f}")

  customer  84  $ 14,673.42
  customer  23  $ 11,703.41
  customer 101  $ 10,591.11
  customer  45  $ 10,533.10
  customer  16  $  9,805.29


### Q3 — Orders per channel per month

Two grouping keys at once — a **composite key** `(month, channel)`. Spark doesn't care what the key is, as long as it hashes consistently.

In [11]:
def parse_order(row):
    oid, cid, d, ch = row
    month = d[:7]                   # 'YYYY-MM-DD' -> 'YYYY-MM'
    return ((month, ch), 1)         # composite key

monthly = (orders
    .map(parse_order)
    .reduceByKey(lambda a, b: a + b))

for (month, ch), n in sorted(monthly.collect()):
    print(f"  {month}  {ch:<10s} {n}")

  2026-01  in_store   118
  2026-01  mobile     119
  2026-01  web        140
  2026-02  in_store   104
  2026-02  mobile     92
  2026-02  web        100
  2026-03  in_store   107
  2026-03  mobile     115
  2026-03  web        105


## Part 4 — Performance patterns

### Broadcast variables — avoid shuffling the big side

`products` is tiny (50 rows); `items` is the large table. Broadcasting `products` means only `items` needs to move (nowhere across the network by key) — no shuffle of the big RDD at all.

In [12]:
prod_map = dict(pid_to_cat.collect())     # tiny -- fits in the driver
prod_bc  = sc.broadcast(prod_map)         # shipped once to every executor

by_cat_bc = (items
    .map(lambda r: (prod_bc.value[int(r[1])], int(r[2]) * float(r[3])))
    .reduceByKey(lambda a, b: a + b))

sorted(by_cat_bc.collect(), key=lambda kv: -kv[1])

[('Electronics', 258738.92999999993),
 ('Books', 209723.77999999997),
 ('Beauty', 198739.59000000008),
 ('Home', 143098.9800000001),
 ('Apparel', 100759.14999999995)]

### `reduceByKey` vs `groupByKey`

Both produce one record per key, but performance is wildly different: `reduceByKey` combines locally on each partition *before* shuffling; `groupByKey` ships every value across the network first.

**Rule of thumb:** if you can express it as `reduceByKey`, never use `groupByKey`.

In [13]:
# GOOD -- partial aggregation happens locally, then partials are summed
good = items.map(lambda r: (int(r[1]), 1)).reduceByKey(lambda a, b: a + b)

# BAD -- ships every single value across the network before summing
bad = items.map(lambda r: (int(r[1]), 1)).groupByKey().mapValues(sum)

# groupByKey is still the right tool when you need *all* the values per key,
# not just an aggregate -- e.g. every order id placed by each customer:
oids_by_cust = (orders
    .map(lambda r: (int(r[1]), r[0]))
    .groupByKey()
    .mapValues(list))

oids_by_cust.take(3)

[(142, ['2', '121', '126', '745']),
 (140, ['3', '340', '425', '861', '913']),
 (130, ['6', '291'])]

### Accumulators — counters across the cluster

Write-only on workers, readable on the driver. Useful for counting skipped/bad rows during a big `map` without a side-channel.

In [14]:
bad_rows = sc.accumulator(0)

def parse_safe(row):
    try:
        return float(row[3])
    except (ValueError, IndexError):
        bad_rows.add(1)
        return 0.0

total = items.map(parse_safe).sum()
print(f"total: {total:.2f}  (bad rows skipped: {bad_rows.value})")

total: 460743.51  (bad rows skipped: 0)


### Caching — avoid recomputing a reused RDD

If you use the same RDD in two or more actions, cache it after the expensive step (parsing, filtering, joining).

In [15]:
cleaned = (items
    .map(lambda r: (int(r[1]), int(r[2]) * float(r[3])))
    .filter(lambda kv: kv[1] > 0))

cleaned.cache()                              # mark for in-memory storage

print("count:", cleaned.count())             # first action: computes + caches
print("total:", cleaned.map(lambda kv: kv[1]).sum())   # second action: reads from cache

count: 2455


total: 911060.4299999999


## Part 5 — DataFrames: MapReduce, modernized

Same computations, expressed with Spark's higher-level, Catalyst-optimized API. The MapReduce shape is still there underneath — `explode` is `flatMap`; `groupBy().count()` is `map(_, 1).reduceByKey(_ + _)`.

In [16]:
from pyspark.sql import functions as F

df = spark.createDataFrame([(l,) for l in [
        "the quick brown fox",
        "the lazy dog",
        "the quick fox jumps over the lazy dog",
    ]], ["line"])

(df.select(F.explode(F.split("line", " ")).alias("word"))
   .groupBy("word")
   .count()
   .orderBy(F.desc("count"))
   .show())

+-----+-----+
| word|count|
+-----+-----+
|  the|    4|
|  fox|    2|
|quick|    2|
| lazy|    2|
|  dog|    2|
|brown|    1|
| over|    1|
|jumps|    1|
+-----+-----+



### DataFrame: revenue by category

Compare to the RDD version above (two re-keys + one join + one reduce) — Catalyst chooses partitioning, join strategy, and broadcast for you.

In [17]:
items_df    = spark.read.csv(f"{ROOT}/items.csv",    header=True, inferSchema=True)
products_df = spark.read.csv(f"{ROOT}/products.csv", header=True, inferSchema=True)
orders_df   = spark.read.csv(f"{ROOT}/orders.csv",   header=True, inferSchema=True)

(items_df
    .join(products_df.select("product_id", "category"), on="product_id")
    .withColumn("line_total", F.col("quantity") * F.col("unit_price"))
    .groupBy("category")
    .agg(F.round(F.sum("line_total"), 2).alias("revenue"))
    .orderBy(F.desc("revenue"))
    .show())

+-----------+---------+
|   category|  revenue|
+-----------+---------+
|Electronics|258738.93|
|      Books|209723.78|
|     Beauty|198739.59|
|       Home|143098.98|
|    Apparel|100759.15|
+-----------+---------+



### DataFrame: top 5 customers by spend

In [18]:
(items_df
    .join(orders_df.select("order_id", "customer_id"), on="order_id")
    .withColumn("line_total", F.col("quantity") * F.col("unit_price"))
    .groupBy("customer_id")
    .agg(F.round(F.sum("line_total"), 2).alias("total_spend"))
    .orderBy(F.desc("total_spend"))
    .show(5))

+-----------+-----------+
|customer_id|total_spend|
+-----------+-----------+
|         84|   14673.42|
|         23|   11703.41|
|        101|   10591.11|
|         45|    10533.1|
|         16|    9805.29|
+-----------+-----------+
only showing top 5 rows


## Exercises

Solve with RDDs first; then redo with DataFrames and compare.

1. **Top product** in each category by units sold.
2. **Repeat customers** — customers who placed orders on more than one date.
3. **Average order value** per channel.
4. **Best day of the week** for revenue (date -> weekday -> group).
5. **Co-purchase pairs** — pairs of products appearing in the same order; top 10 most common pairs.
6. **Customer cohort retention** — per cohort (first-order month), how many customers bought again the following month?

See `mapreduce-pyspark-tutorial.md` for the full write-up and further reading.